# date_parser 수정본 재검증 (300장, cascade 적용)

담당: 이수민

서현이 `integration/ocr-eval` 브랜치의 실제 스크립트(`scripts/evaluate_integrated_baseline.py`,
`evaluate_rotation_retry.py`, `evaluate_highres_retry.py`, `evaluate_contrast_retry.py`)를
그대로 옮겨서 구현한 cascade 노트북입니다. 문서 설명이 아니라 실제 GitHub 코드를 직접 읽고
그 로직을 그대로 따랐습니다.

**채택된 cascade 순서** (서현 `docs/ocr_fallback_experiment_results.md` 최종 pipeline):
1. 원본 512px OCR
2. 날짜 후보가 없을 때만 -> 270도 회전 + 512px OCR
3. 여전히 없을 때만 -> 원본 이미지 1024px OCR
4. 여전히 없을 때만 -> 원본 512px 이미지에 CLAHE 적용 후 OCR
5. 그래도 없으면 원본(512px) 결과를 그대로 사용

각 단계에서 날짜 후보가 생기면 그 즉시 채택하고 다음 단계는 실행하지 않습니다 (조건부 재시도).
"날짜 후보가 있다"의 판정은 서현 스크립트와 동일하게, 각 OCR 박스 텍스트에
`extract_date_tokens()`로 날짜 형태 토큰이 하나라도 있는지로 판단합니다 (정답을 보지 않음).

## 1. 설정

본인 PC 경로에 맞게 필요하면 수정하세요.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / "date_parser").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("date_parser 패키지를 찾을 수 없습니다. itda_OCR 폴더(또는 그 하위)에서 실행하세요.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ===== 본인 환경에 맞게 수정 =====
LABELS_CSV = Path("labels_300.csv")
IMAGE_DIR = Path("label_images")
WEIGHTS_DIR = Path("weights/paddleocr")
OUTPUT_PATH = Path("outputs/validation_cascade_predictions.csv")
MAX_IMAGES = None  # 테스트로 일부만 돌려볼 땐 숫자로, 전체는 None
# =================================

for p, label in [(LABELS_CSV, "라벨 CSV"), (IMAGE_DIR, "이미지 폴더"), (WEIGHTS_DIR, "가중치 폴더")]:
    if not p.exists():
        raise FileNotFoundError(f"{label}을(를) 찾을 수 없습니다: {p.resolve()}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("모든 경로 확인 완료")

PROJECT_ROOT: C:\Users\user\itda_OCR
모든 경로 확인 완료


## 2. date_parser가 최신본인지 확인 (선택, 안전장치)

In [2]:
from date_parser.extract import extract_date_tokens

assert extract_date_tokens("26,07.14"), "date_parser가 예전 버전입니다. extract.py를 최신으로 덮어쓰고 Kernel Restart 하세요."
print("date_parser 최신본 확인 완료")

date_parser 최신본 확인 완료


## 3. 라벨 로드

In [3]:
import pandas as pd

labels = pd.read_csv(LABELS_CSV, dtype=str, encoding="utf-8-sig").fillna("")
labels = labels.loc[:, ~labels.columns.str.startswith("Unnamed")]

required_cols = {"file_name", "image_id", "year", "month", "day", "final_date"}
missing_cols = required_cols - set(labels.columns)
if missing_cols:
    raise ValueError(f"labels_300.csv에 필요한 열이 없습니다: {missing_cols}")

if MAX_IMAGES is not None:
    labels = labels.iloc[:MAX_IMAGES].copy()

print(f"대상: {len(labels)}장")
labels.head()

대상: 300장


,file_name,image_id,year,month,day,final_date,notes
0,000018.jpg,18,2026,4,24,2026-04-24,
1,000026.jpg,26,2025,11,13,2025-11-13,
2,000030.jpg,30,2028,08,17,2028-08-17,
3,000060.jpg,60,2026,04,30,2026-04-30,
4,000075.jpg,75,2026,01,06,2026-01-06,


## 4. 이미지 전처리 함수 (서현 `evaluate_integrated_baseline.py::load_common_input`,
`evaluate_highres_retry.py::load_highres_input` 그대로)

EXIF 방향 보정 -> RGB 변환 -> 긴 변이 `max_side`보다 크면 LANCZOS로 축소 (확대는 안 함).

In [4]:
from PIL import Image, ImageOps
import numpy as np


def load_common_input(path: Path, max_side: int) -> np.ndarray:
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
    long_side = max(image.size)
    if long_side > max_side:
        scale = max_side / long_side
        size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(size, Image.Resampling.LANCZOS)
    return np.asarray(image)

## 5. CLAHE 함수 (서현 `evaluate_contrast_retry.py::apply_clahe` 그대로)

LAB 색공간으로 변환해서 L(명도) 채널에만 CLAHE(clipLimit=2.0, tileGridSize=(8,8)) 적용.

In [5]:
import cv2

CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)


def apply_clahe(rgb: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lightness, channel_a, channel_b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID_SIZE)
    corrected = cv2.merge((clahe.apply(lightness), channel_a, channel_b))
    return cv2.cvtColor(corrected, cv2.COLOR_LAB2RGB)

## 6. PaddleOCR 엔진 초기화 (서현 `evaluate_integrated_baseline.py::initialize_engine`,
`evaluate_highres_retry.py::initialize_highres_engine` 그대로)

512px용 엔진 하나, 1024px 고해상도 재시도 전용 엔진 하나 — 서현 스크립트도 별도 인스턴스로 씁니다.

**`enable_mkldnn`만 False로 둡니다** — 이 컴퓨터의 PaddlePaddle 환경에서 `enable_mkldnn=True`가
`NotImplementedError`를 내는 게 실측으로 확인된 유일한 항목이라, 팀 baseline과 다르게 유지합니다.

In [6]:
import os
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

from paddleocr import PaddleOCR

BATCH_SIZE = 6
CPU_THREADS = 4
BOX_THRESHOLD = 0.7


def make_engine(limit_side_len: int) -> PaddleOCR:
    return PaddleOCR(
        lang="korean",
        text_detection_model_name="PP-OCRv5_mobile_det",
        text_detection_model_dir=str(WEIGHTS_DIR / "PP-OCRv5_mobile_det_infer"),
        text_recognition_model_name="korean_PP-OCRv5_mobile_rec",
        text_recognition_model_dir=str(WEIGHTS_DIR / "korean_PP-OCRv5_mobile_rec_infer"),
        text_recognition_batch_size=BATCH_SIZE,
        text_det_limit_side_len=limit_side_len,
        text_det_limit_type="max",
        device="cpu",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        # MKLDNN=True가 이 컴퓨터에서 NotImplementedError를 내는 게 확인돼서 False로 둡니다.
        enable_mkldnn=False,
        cpu_threads=CPU_THREADS,
    )


engine_512 = make_engine(512)
print("512px 엔진 로딩 완료")
engine_1024 = make_engine(1024)
print("1024px 엔진 로딩 완료")

C:\Users\user\AppData\Local\Temp\ipykernel_9240\2039166308.py:12: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  return PaddleOCR(
Creating model: ('PP-OCRv5_mobile_det', 'weights\\paddleocr\\PP-OCRv5_mobile_det_infer', None)
C:\Users\user\anaconda3\envs\itda\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('korean_PP-OCRv5_mobile_rec', 'weights\\paddleocr\\korean_PP-OCRv5_mobile_rec_infer', None)
Creating model: ('PP-OCRv5_mobile_det', 'weights\\paddleocr\\PP-OCRv5_mobile_det_infer', None)


512px 엔진 로딩 완료


Creating model: ('korean_PP-OCRv5_mobile_rec', 'weights\\paddleocr\\korean_PP-OCRv5_mobile_rec_infer', None)


1024px 엔진 로딩 완료


## 7. 공통 형식 변환 (서현 `evaluate_integrated_baseline.py::paddle_to_common` 그대로)

In [7]:
import json
from typing import Any


def paddle_payload(item: Any) -> dict:
    payload = getattr(item, "json", item)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}


def paddle_to_common(predictions: Any) -> list:
    detections = []
    for item in predictions:
        payload = paddle_payload(item)
        texts = payload.get("rec_texts", []) or []
        scores = payload.get("rec_scores", []) or []
        boxes = payload.get("rec_polys", payload.get("dt_polys", [])) or []
        for index, text in enumerate(texts):
            detections.append({
                "text": str(text),
                "confidence": float(scores[index]) if index < len(scores) else 0.0,
                "bbox": np.asarray(boxes[index]).tolist() if index < len(boxes) else [],
            })
    return detections

## 8. Cascade 실행 함수

서현 스크립트의 `run_rotation` / `run_highres` / `run_variant`(clahe)를 하나의 조건부 흐름으로
합친 버전입니다. 날짜 후보가 없을 때만 다음 단계로 넘어가고, 후보가 생기면 그 즉시 멈춥니다.

In [8]:
from date_parser import parse_expiration_date
from date_parser.select import select_final_date
from date_parser.types import TextBox


def has_candidate_simple(detections: list) -> bool:
    """Stage 1 (512 baseline) escalation check, matching
    evaluate_integrated_baseline.py::add_evaluation_fields exactly:
    just "is there any date-shaped substring anywhere"."""
    return any(extract_date_tokens(item["text"]) for item in detections)


def has_candidate_strict(detections: list) -> bool:
    """Stage 2/3/4 (rotation/highres/clahe) own-success check, matching
    evaluate_rotation_retry.py / evaluate_highres_retry.py /
    evaluate_contrast_retry.py::selected_candidate_details exactly: the
    date-shaped substring must actually produce a valid (year/month/day)
    candidate via select_final_date, not just exist as raw text. Their own
    code uses this stricter check (not the stage-1 one) to decide whether a
    retry pass "worked" and whether to escalate further."""
    boxes = [TextBox.from_dict(item) for item in detections]
    return select_final_date(boxes) is not None


def run_ocr(engine: PaddleOCR, image: np.ndarray) -> list:
    return paddle_to_common(engine.predict(image, text_det_box_thresh=BOX_THRESHOLD))


def run_cascade(image_path: Path) -> dict:
    """512 baseline -> (no candidate) 270 rotation -> (no candidate) 1024 highres
    -> (no candidate) CLAHE -> give up, keep the 512 baseline result.
    Escalation checks intentionally differ by stage - see has_candidate_simple
    vs has_candidate_strict above - because that is what the real scripts do."""
    base_image = load_common_input(image_path, max_side=512)
    baseline_detections = run_ocr(engine_512, base_image)
    if has_candidate_simple(baseline_detections):
        return {"method": "original_512", "detections": baseline_detections}

    rotated = np.asarray(Image.fromarray(base_image).rotate(270, expand=True))
    rotation_detections = run_ocr(engine_512, rotated)
    if has_candidate_strict(rotation_detections):
        return {"method": "rotation_270", "detections": rotation_detections}

    highres_image = load_common_input(image_path, max_side=1024)
    highres_detections = run_ocr(engine_1024, highres_image)
    if has_candidate_strict(highres_detections):
        return {"method": "highres_1024", "detections": highres_detections}

    clahe_image = apply_clahe(base_image)
    clahe_detections = run_ocr(engine_512, clahe_image)
    if has_candidate_strict(clahe_detections):
        return {"method": "clahe", "detections": clahe_detections}

    return {"method": "original_no_candidate", "detections": baseline_detections}

## 9. 300장 실행

cascade라서 기본 노트북보다 느립니다 (후보가 안 나온 이미지만 추가 OCR을 더 도니까요).
서현 실측 기준 300장에 약 11~12분 예상됩니다. 20장마다 진행상황이 출력됩니다.

**중간에 멈춘 것처럼 보여도 Interrupt/정지 누르지 마세요.**

In [9]:
import time

rows = []
debug_rows = []
missing_images = []
start = time.time()

for position, (_, row) in enumerate(labels.iterrows(), start=1):
    image_path = IMAGE_DIR / row["file_name"]
    if not image_path.is_file():
        missing_images.append(row["file_name"])
        rows.append({"image_id": row["image_id"], "year": "NONE", "month": "NONE", "day": "NONE", "final_date": "NONE"})
        debug_rows.append({"image_id": row["image_id"], "detected_text": "(이미지 파일 없음)", "box_count": 0, "selected_method": "missing_image"})
        continue

    outcome = run_cascade(image_path)
    detections = outcome["detections"]
    result = parse_expiration_date(detections)
    rows.append({"image_id": row["image_id"], **result})
    debug_rows.append({
        "image_id": row["image_id"],
        "detected_text": " | ".join(item["text"] for item in detections),
        "box_count": len(detections),
        "selected_method": outcome["method"],
    })

    if position % 20 == 0 or position == len(labels):
        elapsed = time.time() - start
        print(f"  [{position}/{len(labels)}] 진행 중... ({elapsed:.1f}초, 장당 평균 {elapsed / position:.2f}초)")

elapsed = time.time() - start
print(f"\n완료: {len(labels)}장, 총 {elapsed:.1f}초 (장당 평균 {elapsed / len(labels):.2f}초)")
if missing_images:
    print(f"이미지 파일을 못 찾은 항목 {len(missing_images)}건: {missing_images[:10]}")

predictions = pd.DataFrame(rows, columns=["image_id", "year", "month", "day", "final_date"])
ocr_debug = pd.DataFrame(debug_rows, columns=["image_id", "detected_text", "box_count", "selected_method"])
predictions.head()

  [20/300] 진행 중... (85.1초, 장당 평균 4.25초)
  [40/300] 진행 중... (188.2초, 장당 평균 4.71초)
  [60/300] 진행 중... (349.2초, 장당 평균 5.82초)
  [80/300] 진행 중... (479.0초, 장당 평균 5.99초)
  [100/300] 진행 중... (563.3초, 장당 평균 5.63초)
  [120/300] 진행 중... (659.6초, 장당 평균 5.50초)
  [140/300] 진행 중... (788.3초, 장당 평균 5.63초)
  [160/300] 진행 중... (893.4초, 장당 평균 5.58초)
  [180/300] 진행 중... (1011.6초, 장당 평균 5.62초)
  [200/300] 진행 중... (1195.5초, 장당 평균 5.98초)
  [220/300] 진행 중... (1313.8초, 장당 평균 5.97초)
  [240/300] 진행 중... (1528.5초, 장당 평균 6.37초)
  [260/300] 진행 중... (1635.1초, 장당 평균 6.29초)
  [280/300] 진행 중... (1777.8초, 장당 평균 6.35초)
  [300/300] 진행 중... (1912.0초, 장당 평균 6.37초)

완료: 300장, 총 1912.0초 (장당 평균 6.37초)


,image_id,year,month,day,final_date
0,18,2026,04,24,2026-04-24
1,26,2025,11,13,2025-11-13
2,30,2028,08,17,2028-08-17
3,60,2026,04,30,2026-04-30
4,75,2026,01,06,2026-01-06


## 10. 저장

In [10]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_PATH.resolve()}")

저장 완료: C:\Users\user\itda_OCR\outputs\validation_cascade_predictions.csv


## 11. labels_300.csv 기준 정확도

In [11]:
import re

def normalize_component(value):
    value = str(value).strip()
    if value.casefold() == "none":
        return "NONE"
    if re.fullmatch(r"[0-9]+", value):
        return value.lstrip("0") or "0"
    return value

FIELDS = ["year", "month", "day"]

truth = labels[["image_id", *FIELDS]].copy()
truth["image_id"] = truth["image_id"].astype(str).str.strip()
for f in FIELDS:
    truth[f] = truth[f].map(normalize_component)

pred_eval = predictions[["image_id", *FIELDS]].copy()
pred_eval["image_id"] = pred_eval["image_id"].astype(str).str.strip()
for f in FIELDS:
    pred_eval[f] = pred_eval[f].map(normalize_component)

comparison = truth.merge(pred_eval, on="image_id", suffixes=("_true", "_pred"), how="left")
for f in FIELDS:
    comparison[f"{f}_correct"] = comparison[f"{f}_true"] == comparison[f"{f}_pred"]
comparison["final_date_correct"] = comparison[[f"{f}_correct" for f in FIELDS]].all(axis=1)

total = len(comparison)
print(f"총 {total}장")
for f in [*FIELDS, "final_date"]:
    correct = int(comparison[f"{f}_correct"].sum())
    print(f"{f} 정확도: {correct}/{total} = {correct / total:.2%}")

print()
print("채택된 단계별 건수:")
print(ocr_debug["selected_method"].value_counts())

총 300장
year 정확도: 234/300 = 78.00%
month 정확도: 231/300 = 77.00%
day 정확도: 228/300 = 76.00%
final_date 정확도: 215/300 = 71.67%

채택된 단계별 건수:
selected_method
original_512             228
original_no_candidate     40
highres_1024              18
rotation_270               9
clahe                      5
Name: count, dtype: int64


## 12. 실패 사례

In [12]:
failures = comparison.loc[~comparison["final_date_correct"]].merge(
    labels[["image_id", "file_name", "final_date", "notes"]] if "notes" in labels.columns else labels[["image_id", "file_name", "final_date"]],
    on="image_id", how="left"
).merge(ocr_debug, on="image_id", how="left")
print(f"실패 {len(failures)}건")
failures[["image_id", "file_name", "year_true", "year_pred", "month_true", "month_pred",
          "day_true", "day_pred", "selected_method", "detected_text"]]

실패 85건


,image_id,file_name,year_true,year_pred,month_true,month_pred,day_true,day_pred,selected_method,detected_text
0,157,000157.jpg,2027,NONE,8,NONE,7,NONE,original_no_candidate,
1,245,000245.jpg,2026,2025,2,9,24,25,original_512,공 | 소비기한 | 후면표기일까지 | 25.09.25/21:58 | 26.02.24...
2,352,000352.jpg,2026,NONE,8,NONE,27,NONE,original_no_candidate,bo | 보 | 80d | 1800m990 | 이탄 | 269 26% | 0g 0...
3,438,000438.jpg,2026,NONE,2,NONE,19,NONE,original_no_candidate,
4,447,000447.jpg,2025,NONE,11,NONE,22,NONE,original_no_candidate,
...,...,...,...,...,...,...,...,...,...,...
80,2927,002927.jpg,NONE,NONE,4,NONE,8,NONE,original_no_candidate,| TOPPING | 다크츠코 | 토핑부터특별하니까 | CRREEIO | 2074...
81,3228,003228.jpg,2022,NONE,4,NONE,26,NONE,original_no_candidate,FE:0008048
82,3277,003277.jpg,NONE,NONE,4,NONE,5,NONE,original_no_candidate,떠먹는 | SEU | 불기리스 | 토핑과 요거트의 환상 비울로 맛도 | 또떠불 | ...
83,3313,003313.jpg,2021,NONE,7,NONE,5,NONE,original_512,"COCON | 제품명 | Thitinan Food Co.,Ld | 및소제지 | |..."


## 13. 실패 원인 자동 분류

In [13]:
def normalize_digits(value):
    return re.sub(r"[^0-9]", "", str(value))


def expected_date_variants(year, month, day):
    try:
        year, month, day = int(year), int(month), int(day)
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }


def date_recalled(text, year, month, day):
    variants = expected_date_variants(year, month, day)
    if not variants:
        return None
    digits = normalize_digits(text)
    return any(v in digits for v in variants)


def classify_failure(row):
    if row["box_count"] == 0:
        return "OCR 미탐지"
    recalled = date_recalled(row["detected_text"], row["year_true"], row["month_true"], row["day_true"])
    if recalled is None:
        return "정답에 NONE 포함(별도 검토)"
    if recalled is False:
        return "OCR 인식 실패"
    tokens = extract_date_tokens(row["detected_text"])
    if not tokens:
        return "파서 미추출"
    return "파서 선택/해석 오류"


failures["failure_category"] = failures.apply(classify_failure, axis=1)
failures["failure_category"].value_counts()

failure_category
OCR 인식 실패             49
파서 선택/해석 오류           14
정답에 NONE 포함(별도 검토)    14
OCR 미탐지                6
파서 미추출                 2
Name: count, dtype: int64

## 14. 고칠 수 있는 실패만 따로 보기 + 저장

In [14]:
fixable = failures.loc[failures["failure_category"].isin(["파서 미추출", "파서 선택/해석 오류"])]
print(f"date_parser가 고칠 수 있는 실패: {len(fixable)}건")
display(fixable[["image_id", "file_name", "final_date", "year_pred", "month_pred", "day_pred",
         "selected_method", "detected_text", "failure_category"]])

failures.to_csv("outputs/cascade_failures_debug.csv", index=False, encoding="utf-8-sig")
print("저장 완료: outputs/cascade_failures_debug.csv")

date_parser가 고칠 수 있는 실패: 16건


,image_id,file_name,final_date,year_pred,month_pred,day_pred,selected_method,detected_text,failure_category
1,245,000245.jpg,2026-02-24,2025,9,25,original_512,공 | 소비기한 | 후면표기일까지 | 25.09.25/21:58 | 26.02.24...,파서 선택/해석 오류
12,729,000729.jpg,2026-01-07,NONE,NONE,NONE,original_512,"1장국 | 소비기한 | 202.6,01.07 | 04:55 B | AL | 까지 |...",파서 선택/해석 오류
16,922,000922.jpg,2022-02-26,2022,2,NONE,highres_1024,RUC로 | PAPER | 가이스페이피 | EXP:2022/02 | ‘26 | AU,파서 선택/해석 오류
19,1321,001321.jpg,2022-02-15,2022,2,NONE,original_512,란쿠키 | 2022.02.15812:25 | 과자 | 주대산후드/경기도 파주시 | ...,파서 선택/해석 오류
20,1380,001380.jpg,2021-04-09,2021,4,NONE,original_512,20 | T080 15 | - | U합상 | 55 | | B | 2021.04. ...,파서 선택/해석 오류
24,1780,001780.jpg,2020-09-01,2020,3,2,original_512,"포장재질 폴리프로필렌 | 국산), 식물성유지 1[옥수수 외국산(러시아, 형가리,세 ...",파서 선택/해석 오류
26,2011,002011.jpg,2020-06-30,NONE,NONE,NONE,original_no_candidate,2020.C6.30,파서 미추출
30,2162,002162.jpg,2023-06-18,NONE,NONE,NONE,original_512,KIRKLAND | Spanish Quesn | Olives | stuffed wi...,파서 선택/해석 오류
31,2247,002247.jpg,2021-09-07,2020,9,21,original_512,293 | 아이밀 | 남남 | 단호박볼 | J0785F08 | 유통기한: | 20 ...,파서 선택/해석 오류
32,2248,002248.jpg,2021-09-07,2020,9,21,original_512,아이밀 | J0785F08 | 유통기한 | 단호박볼 | 20 21.09.07 | 까지,파서 선택/해석 오류


저장 완료: outputs/cascade_failures_debug.csv
